In [1]:
!pip install -q google-genai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -q python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
import json
import re

from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

env_path = Path.cwd() / ".env"

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY nije pronaden u .env datoteci")

client = genai.Client(api_key=GEMINI_API_KEY)

STUDENT_MODEL = "gemini-3.5-flash-lite"
JUDGE_MODEL = "gemini-3.5-flash"

print("Gemini client is ready.")

Gemini client is ready.


In [9]:
response = client.models.generate_content(
    model=STUDENT_MODEL,
    contents="Reply with word OK."
)

print(response.text)

OK


In [10]:
TEST_CASES = [
    {
        "html": """
        <div class="product" id="P101">
            <h2>Wireless Mouse</h2>
            <span class="price">$24.99</span>
            <span class="stock">In Stock</span>
        </div>
        """,
        "expected": {
            "products": [
                {
                    "product_id": "P101",
                    "product_name": "Wireless Mouse",
                    "price_usd": 24.99,
                    "is_in_stock": True
                }
            ]
        }
    },

    {
        "html": """
        <article class="item" id="P205">
            <p class="stock">Out of Stock</p>
            <span class="name">Mechanical Keyboard</span>
            <span class="price">USD 89.50</span>
        </article>
        """,
        "expected": {
            "products": [
                {
                    "product_id": "P205",
                    "product_name": "Mechanical Keyboard",
                    "price_usd": 89.50,
                    "is_in_stock": False
                }
            ]
        }
    },

    {
        "html": """
        <section class="product" id="P330">
            <strong class="title">USB-C Hub</strong>
            <div data-price="39.90">$39.90 USD</div>
            <div class="availability">In Stock</div>
        </section>
        """,
        "expected": {
            "products": [
                {
                    "product_id": "P330",
                    "product_name": "USB-C Hub",
                    "price_usd": 39.90,
                    "is_in_stock": True
                }
            ]
        }
    },

    {
        "html": """
        <div class="catalog">
            <div class="product" id="P410">
                <h3>Laptop Stand</h3>
                <span>$45.00</span>
                <span>In Stock</span>
            </div>

            <div class="product" id="P411">
                <h3>Web Camera</h3>
                <span>$72.25</span>
                <span>Out of Stock</span>
            </div>
        </div>
        """,
        "expected": {
            "products": [
                {
                    "product_id": "P410",
                    "product_name": "Laptop Stand",
                    "price_usd": 45.00,
                    "is_in_stock": True
                },
                {
                    "product_id": "P411",
                    "product_name": "Web Camera",
                    "price_usd": 72.25,
                    "is_in_stock": False
                }
            ]
        }
    }
]

print("Broj testnih primjera:", len(TEST_CASES))

Broj testnih primjera: 4


In [11]:
INITIAL_PROMPT = """
Extract product information from the provided HTML.

Return the result as JSON.

For every product extract:
- product_id
- product_name
- price_usd
- is_in_stock

Do not include explanations.
"""

print(INITIAL_PROMPT)


Extract product information from the provided HTML.

Return the result as JSON.

For every product extract:
- product_id
- product_name
- price_usd
- is_in_stock

Do not include explanations.



In [12]:
def run_student(system_prompt, html):
    response = client.models.generate_content(
        model=STUDENT_MODEL,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0
        ),
        contents=html
    )

    return response.text.strip()

In [13]:
test_output = run_student(
    INITIAL_PROMPT,
    TEST_CASES[0]["html"]
)

print(test_output)

```json
[
  {
    "product_id": "P101",
    "product_name": "Wireless Mouse",
    "price_usd": 24.99,
    "is_in_stock": true
  }
]
```


In [15]:
def parse_json_output(text):
    text = text.strip()

    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [21]:
def evaluate_output(expected, predicted):
    if predicted is None:
        return {
            "score": 0.0,
            "errors": ["Output nije valjani JSON."]
        }

    if "products" not in predicted:
        return {
            "score": 0.0,
            "errors": ["Nedostaje kljuc 'products'."]
        }

    expected_products = expected["products"]
    predicted_products = predicted["products"]

    if len(expected_products) != len(predicted_products):
        return {
            "score": 0.0,
            "errors": [
                f"Ocekivano proizvoda: {len(expected_products)}, "
                f"dobiveno: {len(predicted_products)}."
            ]
        }

    fields = [
        "product_id",
        "product_name",
        "price_usd",
        "is_in_stock"
    ]

    correct = 0
    total = len(expected_products) * len(fields)
    errors = []

    for i, expected_product in enumerate(expected_products):
        predicted_product = predicted_products[i]

        for field in fields:
            if field not in predicted_product:
                errors.append(
                    f"Proizvod {i + 1}: nedostaje '{field}'."
                )
                continue

            expected_value = expected_product[field]
            predicted_value = predicted_product[field]

            if predicted_value == expected_value:
                correct += 1
            else:
                errors.append(
                    f"Proizvod {i + 1}, {field}: "
                    f"ocekivano {expected_value}, "
                    f"dobiveno {predicted_value}"
                )

    return {
        "score": correct / total,
        "errors": errors
    }

In [22]:
def evaluate_prompt(prompt):
    results = []

    for i, case in enumerate(TEST_CASES, start=1):
        output_text = run_student(
            prompt,
            case["html"]
        )

        parsed_output = parse_json_output(output_text)

        evaluation = evaluate_output(
            case["expected"],
            parsed_output
        )

        results.append({
            "case": i,
            "output": output_text,
            "score": evaluation["score"],
            "errors": evaluation["errors"]
        })

    average_score = sum(
        result["score"] for result in results
    ) / len(results)

    return average_score, results

In [23]:
score, results = evaluate_prompt(INITIAL_PROMPT)

print("Average score:", f"{score:.2%}")

for result in results:
    print("\nTEST", result["case"])
    print("Score:", f"{result['score']:.2%}")
    print("Errors:", result["errors"])

Average score: 0.00%

TEST 1
Score: 0.00%
Errors: ["Nedostaje kljuc 'products'."]

TEST 2
Score: 0.00%
Errors: ["Nedostaje kljuc 'products'."]

TEST 3
Score: 0.00%
Errors: ["Nedostaje kljuc 'products'."]

TEST 4
Score: 0.00%
Errors: ["Nedostaje kljuc 'products'."]


In [26]:
def run_judge(current_prompt, results):
    results_text = ""

    for result in results:
        results_text += f"""
TEST CASE {result['case']}

Student output:
{result['output']}

Score:
{result['score']}

Errors:
{result['errors']}

-----------------------------
"""

    judge_prompt = f"""
You are a prompt optimization expert.

A smaller language model is extracting product information
from HTML into JSON.

The required output format is:

{{
  "products": [
    {{
      "product_id": "string",
      "product_name": "string",
      "price_usd": 0.0,
      "is_in_stock": true
    }}
  ]
}}

Evaluation rules:

1. Output must be valid JSON.
2. The top-level JSON object MUST contain the key "products".
3. "products" MUST contain a list of all products found in the HTML.
4. product_id must come from the HTML id attribute.
5. product_name must contain the correct product name.
6. price_usd must be a numeric value without currency symbols.
7. "In Stock" means true.
8. "Out of Stock" means false.
9. Do not include explanations outside the JSON.

CURRENT SYSTEM PROMPT:

{current_prompt}

STUDENT RESULTS:

{results_text}

Analyze the errors and create a better system prompt
for the smaller model.

Return ONLY valid JSON in this format:

{{
  "optimized_prompt": "the improved system prompt",
  "reasoning": "short explanation of the changes"
}}
"""

    response = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=judge_prompt
    )

    parsed = parse_json_output(response.text)

    if parsed is None:
        raise ValueError("Judge nije vratio valjani JSON.")

    return (
        parsed["optimized_prompt"],
        parsed["reasoning"]
    )

In [27]:
def optimize_prompt(initial_prompt, max_iterations=5):
    current_prompt = initial_prompt
    history = []

    for iteration in range(1, max_iterations + 1):

        print("\n" + "=" * 60)
        print(f"ITERATION {iteration}")
        print("=" * 60)

        average_score, results = evaluate_prompt(current_prompt)

        print("\nAverage score:", f"{average_score:.2%}")

        for result in results:
            print(f"\nTEST {result['case']}")
            print("Score:", f"{result['score']:.2%}")

            if result["errors"]:
                print("Errors:")
                for error in result["errors"]:
                    print("-", error)
            else:
                print("No errors.")

        history.append({
            "iteration": iteration,
            "prompt": current_prompt,
            "score": average_score
        })

        if average_score == 1.0:
            print("\nPerfect score reached.")
            break

        print("\nJudge is optimizing the prompt...")

        new_prompt, reasoning = run_judge(
            current_prompt,
            results
        )

        print("\nJudge reasoning:")
        print(reasoning)

        print("\nNew optimized prompt:")
        print(new_prompt)

        if new_prompt.strip() == current_prompt.strip():
            print("\nPrompt did not change. Stopping.")
            break

        current_prompt = new_prompt

    return current_prompt, history

In [28]:
FINAL_PROMPT, HISTORY = optimize_prompt(
    INITIAL_PROMPT,
    max_iterations=5
)


ITERATION 1

Average score: 0.00%

TEST 1
Score: 0.00%
Errors:
- Nedostaje kljuc 'products'.

TEST 2
Score: 0.00%
Errors:
- Nedostaje kljuc 'products'.

TEST 3
Score: 0.00%
Errors:
- Nedostaje kljuc 'products'.

TEST 4
Score: 0.00%
Errors:
- Nedostaje kljuc 'products'.

Judge is optimizing the prompt...

Judge reasoning:
The smaller model failed because it consistently returned a top-level JSON array instead of a JSON object containing the 'products' key. The optimized prompt explicitly enforces the root object structure, provides a concrete schema template, and strictly forbids outputting a list at the root level or using markdown formatting.

New optimized prompt:
Extract product information from the provided HTML and return it as a single valid JSON object.

Your output MUST follow this exact JSON schema. The root structure must be an object containing the "products" key:

{
  "products": [
    {
      "product_id": "string (extracted from the HTML element's id attribute)",
      "

In [29]:
print("\n" + "=" * 60)
print("OPTIMIZATION SUMMARY")
print("=" * 60)

for item in HISTORY:
    print(
        f"Iteration {item['iteration']}: "
        f"{item['score']:.2%}"
    )

print("\nFINAL OPTIMIZED PROMPT:\n")
print(FINAL_PROMPT)


OPTIMIZATION SUMMARY
Iteration 1: 0.00%
Iteration 2: 100.00%

FINAL OPTIMIZED PROMPT:

Extract product information from the provided HTML and return it as a single valid JSON object.

Your output MUST follow this exact JSON schema. The root structure must be an object containing the "products" key:

{
  "products": [
    {
      "product_id": "string (extracted from the HTML element's id attribute)",
      "product_name": "string",
      "price_usd": number (float, e.g., 99.99, without currency symbols),
      "is_in_stock": boolean (true if 'In Stock', false if 'Out of Stock')
    }
  ]
}

CRITICAL RULES:
1. The top-level output must be a JSON object starting with '{' and ending with '}'. Do NOT start the output with a JSON array/list '['.
2. Do not include markdown code block backticks (like ```json) or any explanations. Output ONLY the raw valid JSON.


In [30]:
print("INITIAL PROMPT SCORE:", f"{HISTORY[0]['score']:.2%}")
print("FINAL PROMPT SCORE:", f"{HISTORY[-1]['score']:.2%}")

INITIAL PROMPT SCORE: 0.00%
FINAL PROMPT SCORE: 100.00%
